In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd

from shapely.geometry import LineString, MultiLineString
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from shapely.geometry import box
import matplotlib.colors as mcolors

from dask import delayed, compute
from tqdm import tqdm
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

# ============================
# User settings
# ============================

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")

boco_ds = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/C2_boco_wind_vec.nc"
clust_ds = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/C2_cluster_wind_vec.nc"

output_dir = '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites'

In [ ]:
client = Client(n_workers=12,
    threads_per_worker=1,
    memory_limit=f"{int(5)}GB"
)
client

In [ ]:
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GULLRWF2',
            'GUNNING1',
            'BANGOWF1',
            'BANGOWF2',
            'COLWF01',
            'WOODLWN1',
            'BOCORWF1']

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]

In [ ]:
extent = [147.5, 151, -38.5, -33.5]
lon_min, lon_max, lat_min, lat_max = extent
step=2

In [ ]:
ds = xr.open_mfdataset(boco_ds, chunks='auto', engine='h5netcdf', parallel=True)
ds = ds.chunk({'time': -1, 'lat': 50, 'lon': 50})

# Convert to Australia/Sydney
local_time = (
    pd.DatetimeIndex(ds.time.values)
    .tz_localize("UTC")
    .tz_convert("Australia/Sydney")
)

# Drop tzinfo so xarray can store it
local_time_naive = local_time.tz_localize(None)

# Assign back to dataset
ds = ds.assign_coords(time=local_time_naive)

In [ ]:
def plot_vector_frame(u, v, lat, lon, t, output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/',
               quiver_scale=None, extent=[147.5, 151, -38.5, -33.5], cluster=cluster, highlight_id='BOCORWF1'):
    """
    Plot wind vectors with optional cluster points.
    
    cluster: DataFrame with columns ['ID', 'lat', 'lon']
    highlight_id: specific ID to highlight in red
    """
    speed = np.sqrt(u**2 + v**2)
    plt.figure(figsize=(10,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, lw=1.5)
    ax.add_feature(cfeature.BORDERS, linestyle='-', lw=1.5)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.5)
    
    # Contour of wind speed
    plt.contourf(lon, lat, speed, cmap='cividis', transform=ccrs.PlateCarree())
    plt.colorbar(label='Wind speed (m/s)')
    
    # Quiver vectors
    plt.quiver(lon, lat, u, v, scale=quiver_scale, color='white', transform=ccrs.PlateCarree())
    
    # Plot cluster points
    if cluster is not None:
        # All points in orange
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", alpha=0.6, s=50, label="Wind Farms", zorder=5, transform=ccrs.PlateCarree())
        
        # Highlight one point in red
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'], 
                       color="red", alpha=0.6, s=80, label=f"ID {highlight_id}", zorder=6,
                       transform=ccrs.PlateCarree())

    plt.title(f'Wind vectors at {str(t)}')
    
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    return filename


def make_filename(t, output_dir, prefix="wind"):
    try:
        # If t is datetime-like, use date formatting
        dt_str = np.datetime_as_string(t, unit='m')
        dt_str = dt_str.replace('-', '')[2:8] + '_' + dt_str[11:13] + dt_str[14:16]
    except Exception:
        # If it's just an index/hour, format as hour
        if isinstance(t, (int, np.integer)):
            dt_str = f"hour{t:02d}"
        else:
            dt_str = str(t).replace(":", "").replace(" ", "_")
    
    return os.path.join(output_dir, f"{prefix}_{dt_str}.png")


In [ ]:
# Crop to extent
ds_subset = ds.sel(**{
    'lon': slice(lon_min, lon_max),
    'lat': slice(lat_min, lat_max)
})

ds_subset

In [ ]:
# Lazy hourly mean
hourly_composite =  ds_subset.groupby("time.hour").mean()

# Compute in parallel
with ProgressBar():
    hourly_composite = hourly_composite.compute()
    hourly_composite = hourly_composite.assign_coords(hour=("hour", np.arange(24)))

In [ ]:
# # Write to NetCDF using compute=False
# delayed_obj = hourly_composite.to_netcdf(
#                 "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/hourly_uv_composite.nc",
#                 engine="h5netcdf",
#                 compute=False)

# # Trigger computation with Dask
# with ProgressBar():
#     delayed_obj.compute()

In [ ]:
lat_subset = hourly_composite['lat'].where(
    (hourly_composite['lat'] >= lat_min) & (hourly_composite['lat'] <= lat_max),
    drop=True
    ).values

lon_subset = hourly_composite['lon'].where(
    (hourly_composite['lon'] >= lon_min) & (hourly_composite['lon'] <= lon_max),
    drop=True
    ).values

for i in range(len(hourly_composite.hour)):
    u_sub = hourly_composite['ua100m'].isel(hour=i).sel(
        lat=lat_subset, lon=lon_subset
    ).values[::step, ::step]

    v_sub = hourly_composite['va100m'].isel(hour=i).sel(
        lat=lat_subset, lon=lon_subset
    ).values[::step, ::step]

    # Also subsample lat/lon to match u/v
    lat_sub = lat_subset[::step]
    lon_sub = lon_subset[::step]

    t = hourly_composite.hour[i].values

    plot_vector_frame(u_sub, v_sub, lat_sub, lon_sub, t, output_dir, 200, extent)

In [ ]:
def calc_divergence(ds):
    """
    Compute divergence of u/v on a spherical Earth using xarray.
    ds should be a Dataset with dimensions lat x lon (or time x lat x lon).
    """
    R = 6371000  # Earth radius in meters
    
    # lat_rad shape (lat,)
    lat_rad = np.deg2rad(ds['lat'].values)
    
    dx_1d = np.gradient(ds['lon'].values) * (np.pi/180) * R
    dx2d = dx_1d[None, :] * np.cos(lat_rad[:, None])  # shape (lat, lon)
    
    dy2d = np.gradient(ds['lat'].values) * (np.pi/180) * R
    dy2d = dy2d[:, None]  # shape (lat, 1) to broadcast with u/v

    # Compute divergence
    divergence = (ds['ua100m'].differentiate('lon') / dx2d +
                  ds['va100m'].differentiate('lat') / dy2d)
    
    return divergence

In [ ]:
def plot_frame(values, lat, lon, title=None,
                          cluster=cluster,
                          highlight_id='BOCORWF1',
                          shapefile_gdf=None,
                          fpath='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/test.png',
                          extent=[147.5, 151, -38.5, -33.5]):

    fig, ax = plt.subplots(figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)
    
    # Divergence field
    cf = ax.contourf(lon, lat, values, cmap='viridis', levels=21, extend='both',
                     transform=ccrs.PlateCarree(), zorder=1)
    plt.colorbar(cf, ax=ax, label='Wind speed')
    
    # --- Contours (from shapefile) ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor='none', edgecolor='black',
                              linewidth=0.5, zorder=3)
    
    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor='black', s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor='black', s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())
    
    plt.title(f'{title}')
    
    # Save
    plt.savefig(fpath, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return fpath


In [ ]:
def plot_divergence_frame(divergence, lat, lon, t, cluster=cluster, highlight_id='BOCORWF1',
                          shapefile_gdf=None,
                          output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/divergence_composite',
                          extent=[147.5, 151, -38.5, -33.5]):
    """
    Plot scalar divergence with optional cluster points and contour shapefile.
    """
    fig, ax = plt.subplots(figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    # Base map
    ax.add_feature(cfeature.COASTLINE, lw=1.0)
    ax.add_feature(cfeature.STATES, linestyle='-', lw=1.0)
    
    # Divergence field
    # Symmetric normalization around zero
    norm = mcolors.TwoSlopeNorm(vmin=np.nanmin(divergence),
                                vcenter=0,
                                vmax=np.nanmax(divergence))
    
    cf = ax.contourf(lon, lat, divergence, cmap='coolwarm', levels=21, norm=norm,
                     transform=ccrs.PlateCarree(), zorder=1)
    plt.colorbar(cf, ax=ax, label='Divergence (1/s)')
    
    # --- Contours (from shapefile) ---
    if shapefile_gdf is not None and not shapefile_gdf.empty:
        for geom in shapefile_gdf.geometry:
            ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                              facecolor='none', edgecolor='black',
                              linewidth=0.5, zorder=3)
    
    # Cluster points
    if cluster is not None:
        ax.scatter(cluster['lon'], cluster['lat'], color="orange", edgecolor='black', s=50,
                   label="Wind Farms", zorder=4, transform=ccrs.PlateCarree())
        if highlight_id is not None and highlight_id in cluster['DUID'].values:
            highlight_point = cluster.loc[cluster['DUID'] == highlight_id]
            ax.scatter(highlight_point['lon'], highlight_point['lat'],
                       color="red", edgecolor='black', s=60,
                       label=f"ID {highlight_id}", zorder=5, transform=ccrs.PlateCarree())
    
    plt.title(f'Composite of divergence at hour {str(t)}')
    
    # Save
    filename = make_filename(t, output_dir)
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return filename

In [ ]:
# Load shapefile
gdf = gpd.read_file('/g/data/ng72/ms5578/ID_HW_BARRA/data/raw/contours/aus25cgd_l.shp').to_crs(epsg=4326)
bbox = box(147.5, -38.5, 151, -33.5)
gdf_clip = gdf[gdf.geometry.intersects(bbox)]

In [ ]:
def okubo_weiss(ds, u_name="ua100m", v_name="va100m", lat_name="lat", lon_name="lon"):
    R = 6371000.0
    lat_vals = ds[lat_name].values
    lon_vals = ds[lon_name].values
    lat_rad = np.deg2rad(lat_vals)

    dlat = np.gradient(lat_vals) * np.pi/180 * R
    dlon = np.gradient(lon_vals) * np.pi/180 * R

    # dx/dy 2D arrays matching full grid
    dx2d = xr.DataArray(dlon[None, :] * np.cos(lat_rad[:, None]),
                        dims=[lat_name, lon_name],
                        coords={lat_name: lat_vals, lon_name: lon_vals})
    dy2d = xr.DataArray(dlat[:, None] * np.ones(len(lon_vals)),
                        dims=[lat_name, lon_name],
                        coords={lat_name: lat_vals, lon_name: lon_vals})

    du_dlon = ds[u_name].differentiate(lon_name) * np.pi/180
    du_dlat = ds[u_name].differentiate(lat_name) * np.pi/180
    dv_dlon = ds[v_name].differentiate(lon_name) * np.pi/180
    dv_dlat = ds[v_name].differentiate(lat_name) * np.pi/180

    dudx = du_dlon / dx2d
    dudy = du_dlat / dy2d
    dvdx = dv_dlon / dx2d
    dvdy = dv_dlat / dy2d

    s_n = dudx - dvdy
    s_s = dudy + dvdx
    omega = dvdx - dudy
    OW = s_n**2 + s_s**2 - omega**2

    return xr.Dataset({"s_n": s_n, "s_s": s_s, "omega": omega, "OW": OW})


In [ ]:
output_dir = '/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_hourly_composites/divergence_composite'

# Subset dataset
ds_subset = hourly_composite.sel(
    lon=slice(lon_min, lon_max),
    lat=slice(lat_min, lat_max)
    )

lat_subset = ds_subset['lat'].values
lon_subset = ds_subset['lon'].values

# Loop over each time step
for i in range(len(ds_subset['hour'])):
    ds_time = ds_subset.isel(hour=i)
    
    # Compute divergence for this timestep
    div = calc_divergence(ds_time)
    
    lat_t = div['lat'].values
    lon_t = div['lon'].values
    div_t = div.values
    
    t = ds_time['hour'].values
    
    filename1 = plot_divergence_frame(div_t, lat_t, lon_t, t,
                                 cluster=cluster,
                                 shapefile_gdf=gdf)

    # calc okubo weiss as well
    ow_ds = okubo_weiss(ds_time, u_name="ua100m", v_name="va100m")
    OW = ow_ds['OW']
    OW_plot = OW.where(~np.isnan(OW))
    
    filename2 = plot_divergence_frame(OW_plot.values, lat_t, lon_t, t, cluster=cluster, shapefile_gdf=gdf, output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/okubo-weiss')
    
    print(f'Saved divergence plots: {filename1} \n {filename2}')

In [ ]:
# speed = np.sqrt(hourly_composite['ua100m']**2 + hourly_composite['va100m']**2)

# wmax = speed.max(dim='hour')
# wmin = speed.min(dim='hour')
# ampl = (speed.max(dim='hour') - speed.min(dim='hour'))

# lat_t = ampl['lat'].values
# lon_t = ampl['lon'].values

# plot_frame(ampl.values, lat_t, lon_t, cluster=cluster, shapefile_gdf=gdf, title='Composite amplitude of diurnal wind cycle (m/s)',
#           fpath='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/composite_amplitude_diurnal.png')
# plot_frame(wmax.values, lat_t, lon_t, cluster=cluster, shapefile_gdf=gdf, title='Maximum of average hourly wind speed (m/s)',
#           fpath='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/composite_max_hour.png')
# plot_frame(wmin.values, lat_t, lon_t, cluster=cluster, shapefile_gdf=gdf, title='Minimum of average hourly wind speed (m/s)',
#           fpath='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/composite_min_hour.png')

In [ ]:
R = 6371000  # Earth radius in meters
    
# lat_rad shape (lat,)
lat_rad = np.deg2rad(hourly_composite['lat'].values)

# dx along longitude, in meters
dx_1d = np.gradient(hourly_composite['lon'].values) * (np.pi/180) * R
dx2d = dx_1d[None, :] * np.cos(lat_rad[:, None])  # shape (lat, lon)

# dy along latitude, in meters
dy2d = np.gradient(hourly_composite['lat'].values) * (np.pi/180) * R
dy2d = dy2d[:, None]  # shape (lat, 1) to broadcast

# Compute curl (z-component)
curl_z = (hourly_composite['va100m'].differentiate('lon') / dx2d -
          hourly_composite['ua100m'].differentiate('lat') / dy2d)

for t in range(24):
    plot_divergence_frame(curl_z[t,:,:], lat_t, lon_t, t, cluster=cluster, highlight_id='BOCORWF1',
                              shapefile_gdf=gdf_clip,
                              output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/curl',
                              extent=[147.5, 151, -38.5, -33.5])
